In [1]:
# Original Kaggle dataset path from ensemble_co2(6).ipynb execution.
from pathlib import Path
DATASET_ROOT = Path("/kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset")
print("Dataset:", DATASET_ROOT)


Dataset: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset


In [2]:
# ==========================================================
# REMNET Stage 2: same dataset source and original ensemble
# No encoded manifest, extra ZIP input, or energy optimization.
# ==========================================================
import os, sys, subprocess, importlib.util, time, random, copy, hashlib, json, zipfile, platform
from pathlib import Path
from collections import defaultdict

# Set GPU visibility before importing torch or CodeCarbon.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
for module, package in [("timm", "timm"), ("codecarbon", "codecarbon==3.3.2"),
                        ("docx", "python-docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
from importlib.metadata import version as installed_version
if installed_version("codecarbon") != "3.3.2":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "codecarbon==3.3.2"])
import timm, codecarbon
from codecarbon import EmissionsTracker
from docx import Document

SEEDS = [7, 49, 112, 345, 476]
BATCH_SIZE, MAX_EPOCHS, PATIENCE, MIN_DELTA = 32, 10, 3, 1e-4
CARBON_FACTOR = 0.475   # kg CO2e/kWh; fixed global assumption, not Kaggle region
MONITOR_SECONDS, MIN_INFERENCE_REPEATS = 15, 20
OUT = Path("/kaggle/working/REMNET_Stage2")
(OUT / "CodeCarbon_Logs").mkdir(parents=True, exist_ok=True)
(OUT / "Model_Checkpoints").mkdir(exist_ok=True)

# ----------------------------
# Dataset: original Kaggle input path (already attached)
# ----------------------------
search_roots = [DATASET_ROOT]
for archive in sorted(DATASET_ROOT.rglob("*.zip")):
    target = Path("/kaggle/working/remnet_source") / archive.stem
    if not (target / ".extract_complete").exists():
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as z:
            for entry in z.infolist():
                if not (target / entry.filename).resolve().is_relative_to(target.resolve()):
                    raise ValueError(f"Unsafe ZIP entry: {entry.filename}")
            z.extractall(target)
        (target / ".extract_complete").touch()
    search_roots.append(target)

split_dirs = []
for root in search_roots:
    for parent in [root, *sorted((p for p in root.rglob("*") if p.is_dir()), key=str)]:
        children = {p.name.lower(): p for p in parent.iterdir() if p.is_dir()}
        tr, te = children.get("training") or children.get("train"), children.get("testing") or children.get("test")
        if tr and te and {p.name for p in tr.iterdir() if p.is_dir()} == {p.name for p in te.iterdir() if p.is_dir()}:
            split_dirs.append((tr, te))
if not split_dirs:
    raise FileNotFoundError("Training/Testing class folders were not found in the original Kaggle dataset.")
TRAIN_DIR, TEST_DIR = split_dirs[0]
CLASS_NAMES = sorted(p.name for p in TRAIN_DIR.iterdir() if p.is_dir())
CLASS_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
assert len(CLASS_NAMES) == 6, f"Expected 6 classes, got {CLASS_NAMES}"
print("Source Training:", TRAIN_DIR, "\nSource Testing:", TEST_DIR)

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

# Stage 1 cleaning: one representative per identical-image group;
# quarantine groups with conflicting labels, and remove the four
# previously visually confirmed near-duplicates.
# The four exclusions are plain filenames, not an encoded manifest.
visual_exclusions = {
    "Training/metal/metal394.jpg", "Training/paper/paper345.jpg",
    "Training/paper/paper390.jpg", "Training/plastic/plastic137.jpg"
}
by_hash = defaultdict(list)
for partition, root in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in CLASS_NAMES:
        for path in sorted((root / cls).iterdir()):
            if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}:
                by_hash[file_sha256(path)].append({"path": path, "class": cls,
                                      "source": f"{partition}/{cls}/{path.name}",
                                      "original_partition": partition})

cleaned = []
conflicts = 0
for digest, group in by_hash.items():
    if len({item["class"] for item in group}) != 1:
        conflicts += len(group)     # Exclude conflicting-label images entirely.
        continue
    item = min(group, key=lambda x: x["source"])  # Prefer Testing over Training.
    cleaned.append({**item, "sha256": digest})

# The Stage 1 80/20 stratified train/validation split was formed BEFORE
# removing 4 visually confirmed near-duplicates. Sort SHA-256 first to
# reproduce exactly the Stage 1 split without embedding any manifest.
original_test = sorted((x for x in cleaned if x["original_partition"] == "Testing"),
                       key=lambda x: x["source"])
train_pool = sorted((x for x in cleaned if x["original_partition"] == "Training"),
                    key=lambda x: x["sha256"])
if (len(cleaned), len(train_pool), len(original_test), conflicts) != (2910, 2447, 463, 6):
    raise ValueError("Dataset differs from audited Stage 1 source; check Kaggle dataset version. "
                     f"Pre-visual counts: total={len(cleaned)}, pool={len(train_pool)}, "
                     f"test={len(original_test)}, conflicting images={conflicts}.")

training, validation = train_test_split(
    train_pool, test_size=0.2, random_state=42,
    stratify=[x["class"] for x in train_pool])
training = sorted((x for x in training if x["source"] not in visual_exclusions),
                  key=lambda x: x["source"])
validation = sorted((x for x in validation if x["source"] not in visual_exclusions),
                    key=lambda x: x["source"])
assert (len(training), len(validation), len(original_test)) == (1956, 487, 463)
assert len({x["sha256"] for x in training + validation + original_test}) == 2906

split_rows = []
for name, group in [("Training", training), ("Validation", validation), ("Testing", original_test)]:
    for item in group:
        split_rows.append({"split": name, "source_path": item["source"],
                           "class": item["class"], "sha256": item["sha256"]})
split_df = pd.DataFrame(split_rows).sort_values(["split", "source_path"])
split_df.to_csv(OUT / "REMNET_Split_Used.csv", index=False)
split_sha = hashlib.sha256(split_df.to_csv(index=False).encode()).hexdigest()
print("Split: Training",len(training),"Validation",len(validation),"Testing",len(original_test))
# Compact integrity check: the original Stage 1 source, members and
# 1,956/487/463 assignments must be IDENTICAL on every execution.
EXPECTED_SPLIT_SHA = "9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792"
if split_sha != EXPECTED_SPLIT_SHA:
    raise RuntimeError("The rebuilt split differs from frozen Stage 1 image membership.")
print("Stage 1 split integrity verified:", split_sha)

# ----------------------------
# Reproducibility, transforms, and datasets
# ----------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Enable a Kaggle GPU to perform the controlled energy experiment.")
print("GPU:", torch.cuda.get_device_name(0), "| Visible GPUs:", torch.cuda.device_count())

train_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15), transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(), transforms.Normalize((0.5,)*3, (0.5,)*3)])
test_tf = transforms.Compose([
    transforms.Resize((32, 32)), transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3)])

class ImageList(Dataset):
    def __init__(self, rows, transform):
        self.rows, self.transform = rows, transform
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as im:
            image = self.transform(im.convert("RGB"))
        return image, CLASS_TO_ID[row["class"]]

train_ds = ImageList(training, train_tf)
val_ds = ImageList(validation, test_tf)
test_ds = ImageList(original_test, test_tf)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def make_model(name, pretrained):
    n = len(CLASS_NAMES)
    if name == "MobileNetV2":
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT if pretrained else None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == "ResNet50":
        m = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if pretrained else None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "EfficientNet-Lite0":
        m = timm.create_model("efficientnet_lite0", pretrained=pretrained, num_classes=n)
    else:
        raise ValueError(name)
    return m.to(device)

def evaluate_loss(model, loader, criterion):
    model.eval(); total = 0.0
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            total += criterion(logits, labels.to(device)).item() * len(labels)
    return total / len(loader.dataset)

def make_tracker(subfolder, name):
    subfolder.mkdir(parents=True, exist_ok=True)
    # Keep an incomplete previous run separate from the current session.
    existing = subfolder / "emissions.csv"
    if existing.exists():
        existing.rename(subfolder / f"emissions_previous_{time.time_ns()}.csv")
    return EmissionsTracker(project_name=name, output_dir=str(subfolder),
                            output_file="emissions.csv", save_to_file=True,
                            tracking_mode="machine", gpu_ids=[0],
                            measure_power_secs=MONITOR_SECONDS, pue=1.0,
                            log_level="info")

def measured_energy(subfolder):
    csv_file = subfolder / "emissions.csv"
    if not csv_file.exists():
        raise FileNotFoundError(f"CodeCarbon output missing: {csv_file}")
    row = pd.read_csv(csv_file).iloc[-1]
    energy = float(row["energy_consumed"])  # Final cumulative value; never sum rows.
    if not np.isfinite(energy) or energy < 0:
        raise ValueError(f"Invalid CodeCarbon energy: {energy}")
    return energy

# ----------------------------
# Brief Kaggle pilot: training/validation image loading, model forward,
# and an original CodeCarbon CSV. Pilot does not inspect the held-out test set.
# ----------------------------
if not (OUT / "pilot_ok.json").exists():
    set_seed(7)
    pilot_model = make_model("MobileNetV2", pretrained=True).eval()
    batch, _ = next(iter(DataLoader(val_ds, batch_size=2, shuffle=False)))
    pilot_log = OUT / "CodeCarbon_Logs" / "pilot"
    tracker = make_tracker(pilot_log, "REMNET_pilot")
    tracker.start()
    try:
        with torch.no_grad():
            result = pilot_model(batch.to(device))
        torch.cuda.synchronize()
    finally:
        tracker.stop()
    if tuple(result.shape) != (2, 6) or not (pilot_log / "emissions.csv").exists():
        raise RuntimeError("Kaggle pilot failed: model output or CodeCarbon log missing")
    (OUT / "pilot_ok.json").write_text(json.dumps({"split_sha256": split_sha,
        "gpu": torch.cuda.get_device_name(0), "codecarbon_version":installed_version("codecarbon")}, indent=2))
    del pilot_model
    torch.cuda.empty_cache()
    print("Pilot passed: image loading, 6-class output, CodeCarbon CSV.")
else:
    pilot_info = json.loads((OUT / "pilot_ok.json").read_text())
    if pilot_info["split_sha256"] != split_sha:
        raise RuntimeError("Previous pilot used a different split. Remove old outputs or restore split.")
    print("Reusing pilot record for identical split.")

# ----------------------------
# Train each constituent ONCE per seed; use validation-loss checkpoints.
# Reuse matching checkpoint + original training record on rerun.
# ----------------------------
def prepare_constituent(name, seed):
    prefix = f"{name}_seed{seed}"
    ckpt_file = OUT / "Model_Checkpoints" / f"{prefix}.pt"
    record_file = OUT / "Model_Checkpoints" / f"{prefix}_training.json"
    log_dir = OUT / "CodeCarbon_Logs" / prefix / "training"
    if ckpt_file.exists() and record_file.exists() and (log_dir / "emissions.csv").exists():
        saved = json.loads(record_file.read_text())
        if saved.get("split_sha256") != split_sha or saved.get("seed") != seed or saved.get("model") != name:
            raise RuntimeError(f"Existing {name} seed {seed} checkpoint belongs to a different experiment")
        model = make_model(name, pretrained=False)
        obj = torch.load(ckpt_file, map_location="cpu", weights_only=True)
        if obj.get("split_sha256") != split_sha or obj.get("seed") != seed:
            raise RuntimeError("Checkpoint metadata mismatch")
        model.load_state_dict(obj["state_dict"])
        print(f"{prefix}: matching checkpoint and CodeCarbon training record reused; no retraining.", flush=True)
        return model.eval(), saved

    model = make_model(name, pretrained=True)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    best_loss, best_epoch, epochs_run, stale = float("inf"), 0, 0, 0
    best_weights = None
    tracker = make_tracker(log_dir, prefix + "_training")
    torch.cuda.synchronize(); tracker.start(); start = time.perf_counter()
    try:
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            running_loss, seen = 0.0, 0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(images), labels)
                loss.backward(); optimizer.step()
                running_loss += loss.item() * len(labels)
                seen += len(labels)
            val_loss = evaluate_loss(model, val_loader, criterion)
            epochs_run = epoch
            print(f"{name}: Epoch {epoch}/{MAX_EPOCHS} | Training Loss: {running_loss/seen:.4f} | Validation Loss: {val_loss:.4f}", flush=True)
            if val_loss < best_loss - MIN_DELTA:
                best_loss, best_epoch, stale = val_loss, epoch, 0
                best_weights = copy.deepcopy(model.state_dict())
            else:
                stale += 1
            if stale >= PATIENCE:
                break
        model.load_state_dict(best_weights)
        torch.cuda.synchronize()
    finally:
        seconds = time.perf_counter() - start
        tracker.stop()
    energy = measured_energy(log_dir)
    saved = {"model":name,"seed":seed,"split_sha256":split_sha,
             "epochs_run":epochs_run,"best_epoch":best_epoch,"best_val_loss":best_loss,
             "training_energy_kWh":energy,"training_co2_kg":energy*CARBON_FACTOR,
             "training_seconds":seconds,"monitoring_seconds":MONITOR_SECONDS,
             "codecarbon_csv":str(log_dir / "emissions.csv")}
    torch.save({"state_dict":model.state_dict(),"seed":seed,"split_sha256":split_sha,
                "model":name,"best_epoch":best_epoch},ckpt_file)
    record_file.write_text(json.dumps(saved, indent=2))
    return model.eval(), saved

# ----------------------------
# Final REMNET held-out testing: equal-weight soft-voting only.
# Training cost is sum of three independent sessions; inference is measured
# separately over repeated complete evaluations of 463 test images.
# ----------------------------
def evaluate_remnet(seed, models_for_seed):
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    for m in models_for_seed.values(): m.eval()
    # Warm-up is excluded from the inference monitoring window.
    first_batch, _ = next(iter(loader))
    with torch.no_grad():
        for _ in range(2):
            for m in models_for_seed.values(): m(first_batch.to(device))
    torch.cuda.synchronize()
    log_dir = OUT / "CodeCarbon_Logs" / f"REMNET_seed{seed}" / "inference"
    tracker = make_tracker(log_dir, f"REMNET_seed{seed}_inference")
    tracker.start(); torch.cuda.synchronize(); start = time.perf_counter()
    repeats = 0
    try:
        with torch.no_grad():
            # Monitor the unchanged three-model equal-weight inference workload.
            while repeats < MIN_INFERENCE_REPEATS or time.perf_counter() - start < 31:
                for images, _ in loader:
                    images = images.to(device)
                    avg_probs = sum(
                        torch.softmax(m(images), dim=1)
                        for m in models_for_seed.values()
                    ) / 3
                repeats += 1
            torch.cuda.synchronize()
    finally:
        seconds = time.perf_counter() - start
        tracker.stop()
    energy = measured_energy(log_dir)
    print(f"REMNET seed {seed}: Ensemble inference completed | "
          f"{repeats} full passes × {len(test_ds)} images | "
          f"batch {BATCH_SIZE} | CodeCarbon interval {MONITOR_SECONDS}s", flush=True)
    return {"seed":seed,
            "inference_energy_kWh":energy,"inference_co2_kg":energy*CARBON_FACTOR,
            "inference_seconds":seconds,"inference_repetitions":repeats,
            "unique_test_images":463,"batch_size":BATCH_SIZE,
            "monitored_image_evaluations":repeats*463,
            "inference_energy_kWh_per_image":energy/(repeats*463),
            "inference_seconds_per_image":seconds/(repeats*463)}

results = []
for seed in SEEDS:
    print("\n==== REMNET SEED",seed,"====")
    set_seed(seed)
    models_for_seed, train_records = {}, []
    for name in ["MobileNetV2", "ResNet50", "EfficientNet-Lite0"]:
        model, train_info = prepare_constituent(name, seed)
        models_for_seed[name] = model
        train_records.append(train_info)
    # No extra ensemble training; fixed soft voting has no fitted parameters.
    row = evaluate_remnet(seed, models_for_seed)
    row["training_energy_kWh"] = sum(r["training_energy_kWh"] for r in train_records)
    row["training_co2_kg"] = sum(r["training_co2_kg"] for r in train_records)
    row["training_seconds"] = sum(r["training_seconds"] for r in train_records)
    row["additional_fitted_ensemble_energy_kWh"] = 0.0
    row["development_energy_accounting_note"] = "Sum of constituent training sessions; no fitted ensemble. Checkpoint-loading/setup energy is not separately metered."
    row["carbon_factor_kg_per_kWh"] = CARBON_FACTOR
    row["codecarbon_version"] = installed_version("codecarbon")
    row["gpu"] = torch.cuda.get_device_name(0)
    row["monitor_interval_seconds"] = MONITOR_SECONDS
    for r in train_records:
        label = r["model"].replace("-", "_")
        row[f"{label}_training_energy_kWh"] = r["training_energy_kWh"]
        row[f"{label}_training_co2_kg"] = r["training_co2_kg"]
        row[f"{label}_training_seconds"] = r["training_seconds"]
        row[f"{label}_epochs_run"] = r["epochs_run"]
        row[f"{label}_best_epoch"] = r["best_epoch"]
    results.append(row)
    print(f"Seed {seed} completed | "
          f"Training: {row['training_energy_kWh']:.6f} kWh / {row['training_co2_kg']:.6f} kg CO2e / {row['training_seconds']/60:.3f} min | "
          f"Inference: {row['inference_energy_kWh']:.6f} kWh / {row['inference_co2_kg']:.6f} kg CO2e / {row['inference_seconds']/60:.3f} min", flush=True)
    del models_for_seed
    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)

def console_energy_summary(title, energy_key, co2_key, seconds_key, extra_key=None):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)
    header = f"{'Seed':>7} {'CO2 (kg)':>14} {'Energy (kWh)':>17} {'Time (min)':>14}"
    if extra_key:
        header += f" {extra_key:>13}"
    print(header)
    for r in results:
        line = (f"{r['seed']:>7} {r[co2_key]:>14.6f} "
                f"{r[energy_key]:>17.6f} {r[seconds_key]/60:>14.3f}")
        if extra_key:
            line += f" {r[extra_key]:>13}"
        print(line)
    print("-" * 75)
    print(f"Mean CO2 (kg):       {np.mean([r[co2_key] for r in results]):.9f}")
    print(f"Mean Energy (kWh):   {np.mean([r[energy_key] for r in results]):.9f}")
    print(f"Mean Time (min):     {np.mean([r[seconds_key] for r in results])/60:.6f}")

console_energy_summary("FINAL 5-SEED SUMMARY — CONSTITUENT TRAINING (3 CNNs; counted once)",
                       "training_energy_kWh", "training_co2_kg", "training_seconds")
console_energy_summary("FINAL 5-SEED SUMMARY — REMNET ENSEMBLE INFERENCE (measured separately)",
                      "inference_energy_kWh", "inference_co2_kg", "inference_seconds",
                       "inference_repetitions")
for r in results:
    r["combined_energy_kWh"] = r["training_energy_kWh"] + r["inference_energy_kWh"]
    r["combined_co2_kg"] = r["training_co2_kg"] + r["inference_co2_kg"]
    r["combined_seconds"] = r["training_seconds"] + r["inference_seconds"]
console_energy_summary("FINAL 5-SEED SUMMARY — FULL REMNET COST (training + one monitored inference session)",
                       "combined_energy_kWh", "combined_co2_kg", "combined_seconds")
print("\nTraining: ResNet50 + MobileNetV2 + EfficientNet-Lite0, once per seed.")
print("Inference: 463 unique images, batch 32, at least 20 complete repetitions;")
print("           CodeCarbon interval 15 seconds, GPU 0 only.")
print("CO2e: CodeCarbon energy × 0.475 kg CO2e/kWh (fixed assumption).")
print("Note: Full-cost row adds one monitored inference session to training;")
print("      it is not an amortized per-image or per-deployment footprint.")

hardware = {"platform":"Kaggle", "python":platform.python_version(),
            "torch":torch.__version__,"torchvision": __import__("torchvision").__version__,
            "timm":timm.__version__,"codecarbon":installed_version("codecarbon"),
            "gpu":torch.cuda.get_device_name(0), "visible_gpu_count":torch.cuda.device_count(),
            "carbon_factor_kg_per_kWh":CARBON_FACTOR,
            "carbon_factor_note":"Fixed global reference; actual Kaggle datacenter carbon intensity unknown",
            "power_sampling_seconds":MONITOR_SECONDS,"batch_size":BATCH_SIZE,
            "split_sha256":split_sha,"unique_test_images":463,
            "split_protocol":"Exact Stage 1 membership reconstructed: SHA-256 sorting, stratified 80/20 split random_state=42, then four visual exclusions"}
(OUT / "Experiment_Configuration.json").write_text(json.dumps(hardware, indent=2))

report = Document()
report.add_heading("REMNET — Stage 2 Sustainability Summary", 0)
report.add_paragraph("Frozen dataset: 1,956 training, 487 validation, 463 held-out testing images. "
    "Five seeds: 7, 49, 112, 345, 476. REMNET combines the same ResNet50, "
    "MobileNetV2, and EfficientNet-Lite0 checkpoints using equal-weight soft voting.")
report.add_paragraph("The frozen Stage 1 image assignments are reconstructed from the "
    "original Kaggle source and verified by split checksum.")
report.add_paragraph("Measured development energy is the sum of the three constituent "
    "training-and-validation sessions, counted once per seed. There is no additional "
    "fitted ensemble. Checkpoint-loading/setup overhead is not separately metered. "
    "Ensemble inference is measured independently: 463 unique test images, "
    "batch size 32, at least 20 complete passes and at least 31 seconds of monitoring, "
    "using GPU 0 and CodeCarbon's 15-second sampling interval. "
    "Standardized CO2e = measured energy × 0.475 kg/kWh (assumed factor).")
table = report.add_table(rows=1, cols=7)
for cell, label in zip(table.rows[0].cells,
    ["Seed", "Train kWh", "Train CO2e kg", "Train s", "Infer kWh", "Infer CO2e kg", "Infer s"]):
    cell.text = label
for _, r in results_df.iterrows():
    cells = table.add_row().cells
    for cell, value in zip(cells, [str(int(r.seed)), f"{r.training_energy_kWh:.8f}",
                              f"{r.training_co2_kg:.8f}",f"{r.training_seconds:.2f}",
                              f"{r.inference_energy_kWh:.8f}",f"{r.inference_co2_kg:.8f}",
                              f"{r.inference_seconds:.2f}"]):
        cell.text = value
report.add_paragraph("Original CodeCarbon emissions CSV files, constituent checkpoints, "
    "epochs and monitoring records are preserved under REMNET_Stage2.")
report.save(OUT / "Stage2_Summary.docx")
print("\nResults displayed in the Kaggle notebook console.")
print("Original CodeCarbon CSVs, checkpoint evidence, split records and report saved in:", OUT)


Source Training: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Training 
Source Testing: /kaggle/input/datasets/saumyamohandas/garbage-classification-image-dataset/dataset/Testing
Split: Training 1956 Validation 487 Testing 463
Stage 1 split integrity verified: 9fcca4cba347d47e6e01fcb12ffc246e6762a724d3709764c2e2f72e7d5e4792
GPU: Tesla T4 | Visible GPUs: 1
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 154MB/s]
[codecarbon WARNING @ 07:36:04] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 07:36:04] Emissions data (if any) will be saved to file /kaggle/working/REMNET_Stage2/CodeCarbon_Logs/pilot/emissions.csv
[codecarbon INFO @ 07:36:04] [setup] RAM Tracking...
[codecarbon INFO @ 07:36:04] [setup] CPU Tracking...
[codecarbon INFO @ 07:36:04] [setup] GPU Tracking...
[codecarbon INFO @ 07:36:04] Tracking Nvidia GPUs via PyNVML
[codecarbon INFO @ 07:36:04] Monitoring GPUs with indices: [0] out of 2 total GPUs
[codecarbon WARNING @ 07:36:04] You have 2 GPUs but we will monitor only 1 ([0]) of them.
[codecarbon WARNING @ 07:36:05] We saw that you have a Intel(R) Xeon(R) CPU @ 2.00GHz but we don't know it. Please help us add it, see https://docs.codecarbon.io/latest/how-to/troubleshooting/#unknown-cpu-model
[codecarbon WARNING @ 07:36:05] We will use the default power consumption of 4 W per thread for your 

Pilot passed: image loading, 6-class output, CodeCarbon CSV.

==== REMNET SEED 7 ====
MobileNetV2: Epoch 1/10 | Training Loss: 1.7862 | Validation Loss: 1.7164


[codecarbon INFO @ 07:36:25] Energy consumed for RAM : 0.000083 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 07:36:25] Energy consumed for all GPUs : 0.000117 kWh. Total GPU Power : 28.14189665978992 W
[codecarbon INFO @ 07:36:25] Delta energy consumed for CPU with cpu_load : 0.000008 kWh, power : 1.8835588151794052 W
[codecarbon INFO @ 07:36:25] Energy consumed for All CPU : 0.000008 kWh
[codecarbon INFO @ 07:36:25] 0.000208 kWh of electricity and 0.000000 L of water were used since the beginning.


MobileNetV2: Epoch 2/10 | Training Loss: 1.6622 | Validation Loss: 1.6138
MobileNetV2: Epoch 3/10 | Training Loss: 1.5745 | Validation Loss: 1.5325


[codecarbon INFO @ 07:36:40] Energy consumed for RAM : 0.000167 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 07:36:40] Energy consumed for all GPUs : 0.000234 kWh. Total GPU Power : 28.169309223457084 W
[codecarbon INFO @ 07:36:40] Delta energy consumed for CPU with cpu_load : 0.000008 kWh, power : 1.8832827561318148 W
[codecarbon INFO @ 07:36:40] Energy consumed for All CPU : 0.000016 kWh
[codecarbon INFO @ 07:36:40] 0.000417 kWh of electricity and 0.000000 L of water were used since the beginning.


... (188 outputs left out)

[codecarbon INFO @ 07:59:44] Energy consumed for RAM : 0.000500 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 07:59:44] Energy consumed for all GPUs : 0.000853 kWh. Total GPU Power : 34.48423918065963 W
[codecarbon INFO @ 07:59:44] Delta energy consumed for CPU with cpu_load : 0.000008 kWh, power : 1.887454651743271 W
[codecarbon INFO @ 07:59:44] Energy consumed for All CPU : 0.000047 kWh
[codecarbon INFO @ 07:59:44] 0.001400 kWh of electricity and 0.000000 L of water were used since the beginning.


EfficientNet-Lite0: Epoch 10/10 | Training Loss: 1.6137 | Validation Loss: 1.7556


[codecarbon INFO @ 07:59:48] Energy consumed for RAM : 0.000524 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 07:59:48] Energy consumed for all GPUs : 0.000896 kWh. Total GPU Power : 34.7863584028254 W
[codecarbon INFO @ 07:59:48] Delta energy consumed for CPU with cpu_load : 0.000002 kWh, power : 1.8660446354689129 W
[codecarbon INFO @ 07:59:48] Energy consumed for All CPU : 0.000049 kWh
[codecarbon INFO @ 07:59:48] 0.001470 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 07:59:51] 0.002494 g.CO2eq/s mean an estimation of 78.65085230396946 kg.CO2eq/year
[codecarbon WARNING @ 07:59:52] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 07:59:52] Emissions data (if any) will be saved to file /kaggle/working/REMNET_Stage2/CodeCarbon_Logs/REMNET_seed476/inference/emissions.csv
[codecarbon INFO @ 07:59:52] Monitoring GPUs with indices: [0] out of 2 total GPUs
[codecarbon WARNING @ 07:59:52] You have 2 GPUs but 

REMNET seed 476: Ensemble inference completed | 20 full passes × 463 images | batch 32 | CodeCarbon interval 15s
Seed 476 completed | Training: 0.003841 kWh / 0.001824 kg CO2e / 3.970 min | Inference: 0.000530 kWh / 0.000252 kg CO2e / 0.553 min

FINAL 5-SEED SUMMARY — CONSTITUENT TRAINING (3 CNNs; counted once)
   Seed       CO2 (kg)      Energy (kWh)     Time (min)
      7       0.002162          0.004551          4.887
     49       0.001810          0.003810          3.956
    112       0.001727          0.003635          3.780
    345       0.001737          0.003656          3.810
    476       0.001824          0.003841          3.970
---------------------------------------------------------------------------
Mean CO2 (kg):       0.001851850
Mean Energy (kWh):   0.003898631
Mean Time (min):     4.080626

FINAL 5-SEED SUMMARY — REMNET ENSEMBLE INFERENCE (measured separately)
   Seed       CO2 (kg)      Energy (kWh)     Time (min) inference_repetitions
      7       0.000245       

In [3]:
# ================================================================
# ADDITIONAL EXPORT ONLY — held-out test-set probabilities
# Run AFTER the original REMNET Stage 2 cell has finished.
# No changes to training, checkpoint selection, split, monitored
# inference, CodeCarbon measurements, or existing output files.
# This separate export is not included in the energy measurements.
# ================================================================
PROBABILITY_DIR = OUT / "Test_Probabilities"
PROBABILITY_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_MODEL_NAMES = ("MobileNetV2", "ResNet50", "EfficientNet-Lite0")
probability_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)
test_sources = [item["source"] for item in original_test]
test_true_classes = [item["class"] for item in original_test]
test_true_ids = [CLASS_TO_ID[name] for name in test_true_classes]
all_remnet_probabilities = []

for probability_seed in SEEDS:
    loaded_models = {}
    try:
        # Reload the EXACT best-validation checkpoints from the
        # original run; do not train or change any model weights.
        for model_name in EXPORT_MODEL_NAMES:
            checkpoint_path = (
                OUT / "Model_Checkpoints" /
                f"{model_name}_seed{probability_seed}.pt"
            )
            if not checkpoint_path.is_file():
                raise FileNotFoundError(
                    f"Missing original model checkpoint: {checkpoint_path}. "
                    "Run the original Stage 2 cell first."
                )
            checkpoint = torch.load(
                checkpoint_path, map_location="cpu", weights_only=True
            )
            if (
                checkpoint.get("seed") != probability_seed
                or checkpoint.get("split_sha256") != split_sha
                or checkpoint.get("model") != model_name
            ):
                raise RuntimeError(
                    f"Checkpoint identity/split mismatch: {checkpoint_path}"
                )
            current_model = make_model(model_name, pretrained=False)
            current_model.load_state_dict(checkpoint["state_dict"])
            current_model.eval()
            loaded_models[model_name] = current_model

        collected = {
            model_name: [] for model_name in
            (*EXPORT_MODEL_NAMES, "REMNET")
        }

        # Separate, unmonitored prediction-only pass over the
        # unchanged held-out 463-image test set.
        with torch.no_grad():
            for images, _ in probability_loader:
                images = images.to(device)
                component_probabilities = {
                    model_name: torch.softmax(model(images), dim=1)
                    for model_name, model in loaded_models.items()
                }
                # EXACT ensemble rule used in the existing notebook.
                remnet_probabilities = (
                    sum(component_probabilities.values()) / 3
                )
                component_probabilities["REMNET"] = remnet_probabilities

                for model_name, probabilities in component_probabilities.items():
                    collected[model_name].append(
                        probabilities.detach().cpu().numpy()
                    )

        # One separate per-image, per-class CSV for each model and REMNET.
        for model_name, batches in collected.items():
            probability_array = np.concatenate(batches, axis=0)
            expected_shape = (len(original_test), len(CLASS_NAMES))
            if probability_array.shape != expected_shape:
                raise RuntimeError(
                    f"{model_name}: expected {expected_shape}, "
                    f"received {probability_array.shape}"
                )
            if not np.allclose(
                probability_array.sum(axis=1), 1.0, atol=1e-5
            ):
                raise RuntimeError(
                    f"{model_name}: predicted probabilities do not sum to one."
                )
            predicted_ids = probability_array.argmax(axis=1)
            exported_rows = {
                "seed": [probability_seed] * len(original_test),
                "source_path": test_sources,
                "true_class": test_true_classes,
                "true_class_id": test_true_ids,
                "predicted_class": [
                    CLASS_NAMES[int(index)] for index in predicted_ids
                ],
                "predicted_class_id": predicted_ids,
                "prediction_confidence": probability_array.max(axis=1),
            }
            for class_index, class_name in enumerate(CLASS_NAMES):
                exported_rows[f"prob_{class_name}"] = (
                    probability_array[:, class_index]
                )

            probability_frame = pd.DataFrame(exported_rows)
            safe_model_name = model_name.replace("-", "_")
            probability_file = (
                PROBABILITY_DIR /
                f"{safe_model_name}_seed{probability_seed}_test_probabilities.csv"
            )
            probability_frame.to_csv(
                probability_file, index=False, float_format="%.10f"
            )
            print(
                f"Saved {model_name}, seed {probability_seed}: "
                f"{len(probability_frame)} images -> {probability_file}"
            )
            if model_name == "REMNET":
                all_remnet_probabilities.append(probability_frame)
    finally:
        loaded_models.clear()
        torch.cuda.empty_cache()

# Convenient five-seed combined CSV for the REMNET ensemble.
combined_remnet_file = (
    PROBABILITY_DIR / "REMNET_all_seeds_test_probabilities.csv"
)
pd.concat(all_remnet_probabilities, ignore_index=True).to_csv(
    combined_remnet_file, index=False, float_format="%.10f"
)
print("\nProbability export complete. All original energy results unchanged.")
print("Individual models and REMNET (5 seeds each):", PROBABILITY_DIR)
print("Combined REMNET CSV:", combined_remnet_file)


Saved MobileNetV2, seed 7: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/MobileNetV2_seed7_test_probabilities.csv
Saved ResNet50, seed 7: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/ResNet50_seed7_test_probabilities.csv
Saved EfficientNet-Lite0, seed 7: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed7_test_probabilities.csv
Saved REMNET, seed 7: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/REMNET_seed7_test_probabilities.csv
Saved MobileNetV2, seed 49: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/MobileNetV2_seed49_test_probabilities.csv
Saved ResNet50, seed 49: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/ResNet50_seed49_test_probabilities.csv
Saved EfficientNet-Lite0, seed 49: 463 images -> /kaggle/working/REMNET_Stage2/Test_Probabilities/EfficientNet_Lite0_seed49_test_probabilities.csv
Saved REMNET, seed 49: 463 images -> /kaggle/working/REMNET_Stage2/Test